# rvt-rs — Python quickstart

Walk through the most common uses of the `rvt` Python bindings: open a Revit file, inspect metadata, read the root document via the schema-directed walker, and export a document-level IFC4 scaffold.

**Install**: `pip install rvt` once the wheel is published; from a source checkout run `maturin build --release --features python && pip install target/wheels/rvt-*.whl`.

**Sample file**: this notebook uses `racbasicsamplefamily-2024.rfa` from the public `rac_basic_sample_family` corpus Autodesk ships with every Revit release. The repo's `samples/` directory (and the `RVT_SAMPLES_DIR` env var honoured below) is where the integration tests look for it. Any `.rvt`/`.rfa`/`.rte`/`.rft` file works — swap the path in cell 1.

In [ ]:
import os
import pathlib

import rvt

print(f"rvt version: {rvt.__version__}")

# Sample-path resolution mirrors tests/python/test_rvt.py:
#   1. honour the RVT_SAMPLES_DIR env var if set
#   2. otherwise try a handful of sensible paths relative to the
#      notebook's working directory
#
# The in-repo `samples/` directory lives at the monorepo root in the
# canonical checkout (../../../samples when this notebook is run from
# <rvt-rs>/docs/). Point RVT_SAMPLES_DIR at any directory holding
# `racbasicsamplefamily-*.rfa` to use your own corpus.
def resolve_samples_dir() -> pathlib.Path:
    env_dir = os.environ.get("RVT_SAMPLES_DIR")
    if env_dir:
        return pathlib.Path(env_dir).expanduser().resolve()
    here = pathlib.Path.cwd().resolve()
    candidates = [
        here / "samples",
        here.parent / "samples",
        here.parent.parent / "samples",
        here.parent.parent.parent / "samples",
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[-1]

samples_dir = resolve_samples_dir()
SAMPLE = samples_dir / "racbasicsamplefamily-2024.rfa"
print(f"Samples dir : {samples_dir}")
print(f"Sample file : {SAMPLE.name}")
print(f"Exists      : {SAMPLE.exists()}")
if not SAMPLE.exists():
    print()
    print("To run this notebook, set RVT_SAMPLES_DIR to a directory")
    print("containing Autodesk's public `rac_basic_sample_family` corpus,")
    print("or point SAMPLE at any .rvt/.rfa/.rte/.rft file you have locally.")

## 1. Open the file and inspect metadata

Constructing `RevitFile(path)` opens the MS-CFB container and keeps the bytes cached in memory. The metadata getters (`version`, `build`, `guid`, `part_atom_title`, `original_path`) parse on-demand — nothing heavy runs until you call a method that actually touches a stream. Size limits default to 2 GiB file / 256 MiB stream / 256 MiB inflate; override via `max_file_bytes=`, `max_stream_bytes=`, `max_inflate_bytes=` keyword arguments on the constructor if you need to cap the reader tighter for hostile input.

In [ ]:
f = rvt.RevitFile(str(SAMPLE))

print(f"Revit release  : {f.version}")
print(f"Build tag      : {f.build}")
print(f"Document title : {f.part_atom_title}")
print(f"Document GUID  : {f.guid}")
print(f"Original path  : {f.original_path}")
print(f"repr()         : {f!r}")

## 2. Full `BasicFileInfo` as JSON

`basic_file_info_json()` returns the entire `BasicFileInfo` record as a JSON string. It is the single-call equivalent of the four getters above (`version`, `build`, `guid`, `original_path`) plus any future fields the Rust `BasicFileInfo` struct grows — `json.loads()` gives you a plain dict. Returns `None` if the `BasicFileInfo` stream can't be parsed.

In [ ]:
import json

bfi_json = f.basic_file_info_json()
if bfi_json is None:
    print("BasicFileInfo stream unreadable; metadata getters will also return None")
else:
    bfi = json.loads(bfi_json)
    print(json.dumps(bfi, indent=2))

## 3. List the OLE streams

A Revit file is an MS-CFB container holding a set of named streams. The 2016–2026 releases all carry the same 12 invariant streams plus a version-specific `Partitions/NN` marker. `missing_required_streams()` is the cheapest way to pre-validate an input before running heavy extractors — an empty list means the file at least looks structurally like Revit.

In [ ]:
for s in f.stream_names():
    print(f"  {s}")

missing = f.missing_required_streams()
if missing:
    print(f"\nmissing required streams: {missing}")
else:
    print("\nall required streams present")

## 4. Schema inventory

The `Formats/Latest` stream carries Revit's complete serialisation schema: every class name, every field, every C++ type signature. `schema_summary()` is cheap (counts only); `schema_json()` is the full dump and runs around 1–2 MB for a typical family. We print the summary plus the first few class names from the full dump to keep the output readable.

In [ ]:
summary = f.schema_summary()
print(f"Classes in this file   : {summary['classes']}")
print(f"Total declared fields  : {summary['fields']}")
print(f"Distinct C++ type sigs : {summary['cpp_types']}")

schema = json.loads(f.schema_json())
print(f"\nFirst 10 class names (of {len(schema['classes'])}):")
for cls in schema['classes'][:10]:
    print(f"  tag={cls['tag']:>5}  fields={len(cls['fields']):>3}  {cls['name']}")

# Confirm ADocument is present — it is the root class the Layer-5a
# walker targets in the next cell.
adoc = next((c for c in schema['classes'] if c['name'] == 'ADocument'), None)
print(f"\nADocument found: {adoc is not None}  "
      f"(fields={len(adoc['fields']) if adoc else 'n/a'})")

## 5. Read the root `ADocument` record

`read_adocument()` runs the Layer-5a walker and returns `ADocument`'s declared instance fields as a Python dict, or `None` if the entry-point detector can't confidently locate the record. Releases 2024–2026 are cross-version validated: the last three `ElementId` fields (`m_ownerFamilyId`, `m_ownerFamilyContainingGroupId`, `m_devBranchInfo`) are byte-identical across those releases on the reference corpus. Releases 2016–2023 decode the schema in full but the walker currently reports `partial` on them — see `docs/compatibility.md` for the matrix.

Per-field decoding beyond `ADocument` (walls, doors, levels, families, etc.) is implemented on the Rust side; exposing it to Python is tracked separately (see task #147 / `ADocumentInstance` completeness markers).

In [ ]:
doc = f.read_adocument()
if doc is None:
    print("walker couldn't locate ADocument in this file")
else:
    print(f"ADocument at offset 0x{doc['entry_offset']:06x}  (Revit {doc['version']})")
    print()
    for i, field in enumerate(doc['fields']):
        kind = field['kind']
        name = field['name']
        detail = ''
        if kind == 'pointer':
            detail = f"slots [{field['slot_a']:#x}, {field['slot_b']:#x}]"
        elif kind == 'element_id':
            detail = f"tag={field['tag']}, id={field['id']}"
        elif kind == 'ref_container':
            detail = f"count={field['count']}"
        elif kind == 'bytes':
            detail = f"len={field['len']}"
        print(f"  [{i:>2}] {name:<36} {kind:<14}  {detail}")

## 6. Export to IFC4 (document-level)

`write_ifc()` produces a spec-valid IFC4 STEP text: the ISO-10303-21 envelope, the required framework entities (`IfcPerson` / `IfcOrganization` / `IfcApplication` / `IfcOwnerHistory` / `IfcSIUnit` / `IfcUnitAssignment` / `IfcGeometricRepresentationContext`), and an `IfcProject` populated from the document metadata.

**Coverage is document-level only.** Building-element geometry (walls, doors, floors, roofs, etc.) is not yet emitted — that pipeline depends on per-class element decoders currently exposed only through the Rust API. See `docs/compatibility.md` §IFC for the current scope and `CONTRIBUTING.md` → *Where help is most wanted* if you want to extend coverage.

In [ ]:
ifc_text = f.write_ifc()
print(f"IFC output length : {len(ifc_text):,} bytes")
print(f"IFC4 schema       : {'FILE_SCHEMA(' in ifc_text and 'IFC4' in ifc_text}")
print(f"IfcProject count  : {ifc_text.count('IFCPROJECT(')}")

# Write the STEP text next to the input so you can inspect it with
# any IFC viewer / validator (e.g. IfcOpenShell's `IfcConvert`).
out_path = pathlib.Path(str(SAMPLE).replace('.rfa', '.ifc'))
out_path.write_text(ifc_text)
print(f"Wrote             : {out_path}")

print()
print("First 20 lines of the STEP file:")
for line in ifc_text.splitlines()[:20]:
    print(f"  {line}")

## 7. Cross-version sanity check

If you have multiple `racbasicsamplefamily-*.rfa` releases on disk, this cell opens each and prints the reported Revit release year. The 2016–2019 files use the older `rac_basic_sample_family-<year>.rfa` naming; 2020+ use the compact `racbasicsamplefamily-<year>.rfa` form. Skip this cell if you only have the one sample.

In [ ]:
candidates = sorted(samples_dir.glob('rac*sample*family*.rfa')) if samples_dir.exists() else []

if not candidates:
    print(f"No corpus files found under {samples_dir}")
else:
    for path in candidates:
        try:
            g = rvt.RevitFile(str(path))
            print(f"  {path.name:<45} Revit {g.version}  title={g.part_atom_title!r}")
        except Exception as exc:
            print(f"  {path.name:<45} <open failed: {exc}>")

## Where to next?

- **Compatibility matrix**: [`docs/compatibility.md`](compatibility.md) — release-by-release breakdown of what parses vs. what the walker decodes vs. what the IFC exporter emits. Source-of-truth for current limitations.
- **Python API reference**: [`docs/python.md`](python.md) — full surface of the `rvt` module with return shapes.
- **Recon report**: [`docs/rvt-moat-break-reconnaissance.md`](rvt-moat-break-reconnaissance.md) — the living technical record of the Revit on-disk format as we've reverse-engineered it clean-room.
- **Contributing**: [`CONTRIBUTING.md`](../CONTRIBUTING.md) — especially the *Where help is most wanted* section if you want to extend the walker or IFC coverage.
- **Issues**: [github.com/DrunkOnJava/rvt-rs/issues](https://github.com/DrunkOnJava/rvt-rs/issues)

### Current limitations (summary)

- **Walker coverage**: `ADocument` root fields decoded across all 11 releases; fully cross-version-validated on 2024–2026. Per-class element decoders (walls, doors, families, etc.) exist on the Rust side but are not yet exposed through the Python bindings.
- **IFC export**: document-level scaffold only. No element geometry, no materials, no property sets yet. Output is schema-valid IFC4 STEP that `IfcOpenShell` will round-trip.
- **Write path**: the Rust crate supports a verified write-back round-trip; no Python binding yet.